# EarthScape Climate Agency
## Big Data & Machine Learning Climate Analytics Notebook
**Project:** EarthScape Climate Monitoring and Analytics Platform  
**Environment:** Jupyter Anaconda Notebook 3 / Python 3.10+  
**Objective:** Ingest climate datasets, perform Exploratory Data Analysis (EDA), detect climate anomalies, and train predictive machine learning models for climate trend forecasting.

### 1. Import Required Libraries

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import Ridge, LinearRegression
from sklearn.preprocessing import PolynomialFeatures
from sklearn.pipeline import make_pipeline
from sklearn.ensemble import IsolationForest
from sklearn.metrics import mean_squared_error, r2_score

# Set visualization aesthetics
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['figure.figsize'] = (12, 6)
print("Libraries successfully loaded!")

### 2. Load Climate Datasets (Historical & Anomaly Benchmark)

In [ ]:
data_path_normal = os.path.join('..', 'data', 'sample', 'normal_data.csv')
data_path_anomaly = os.path.join('..', 'data', 'sample', 'anomaly_data.csv')

df_normal = pd.read_csv(data_path_normal)
df_anomaly = pd.read_csv(data_path_anomaly)

# Combine datasets for unified exploration
df = pd.concat([df_normal, df_anomaly], ignore_index=True)
df['date'] = pd.to_datetime(df['date'])
df = df.sort_values(['location', 'date']).reset_index(drop=True)

print(f"Total climate records loaded: {len(df):,}")
df.head()

### 3. Summary Statistics & Data Profiling

In [ ]:
print("Summary Statistics of Climate Metrics:")
display(df[['temperature', 'humidity', 'rainfall', 'wind_speed', 'air_pressure', 'co2']].describe())

### 4. Correlation Analysis (Pearson Correlation Matrix)

In [ ]:
features = ['temperature', 'humidity', 'rainfall', 'wind_speed', 'air_pressure', 'co2']
corr_matrix = df[features].corr(method='pearson')

plt.figure(figsize=(9, 7))
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', fmt=".2f", cbar=True, square=True)
plt.title('EarthScape Climate Variables Correlation Matrix (Pearson)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

### 5. Temperature Distribution by Geographic Station

In [ ]:
plt.figure(figsize=(14, 6))
sns.boxplot(x='location', y='temperature', data=df, palette='Spectral')
plt.title('Ambient Temperature Distribution across Weather Stations', fontsize=14, fontweight='bold')
plt.xlabel('Station Location')
plt.ylabel('Temperature (°C)')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

### 6. Anomaly Detection using Isolation Forest & Statistical Z-Score

In [ ]:
# Multi-dimensional Isolation Forest
iso_forest = IsolationForest(contamination=0.03, random_state=42)
df['anomaly_iso'] = iso_forest.fit_predict(df[features])

# Statistical Z-Score for temperature
temp_mean = df['temperature'].mean()
temp_std = df['temperature'].std()
df['z_temp'] = (df['temperature'] - temp_mean) / temp_std
df['anomaly_zscore'] = df['z_temp'].abs() > 2.5

total_iso = (df['anomaly_iso'] == -1).sum()
total_z = df['anomaly_zscore'].sum()
print(f"Anomalies identified by Isolation Forest: {total_iso}")
print(f"Anomalies identified by Z-Score (>2.5 std): {total_z}")

# Visualize detected anomalies for a sample station
sample_loc = df['location'].unique()[0]
sub_df = df[df['location'] == sample_loc].sort_values('date')

plt.figure(figsize=(14, 6))
plt.plot(sub_df['date'], sub_df['temperature'], label='Normal Temperature', color='steelblue', alpha=0.7)
anom_pts = sub_df[sub_df['anomaly_iso'] == -1]
plt.scatter(anom_pts['date'], anom_pts['temperature'], color='crimson', s=60, label='Detected Anomaly', zorder=5)
plt.title(f'Climate Anomaly Detection for {sample_loc} Station', fontsize=14, fontweight='bold')
plt.xlabel('Date')
plt.ylabel('Temperature (°C)')
plt.legend()
plt.tight_layout()
plt.show()

### 7. Predictive Machine Learning (Polynomial Time-Series Ridge Regressor)

In [ ]:
# Prepare sequential day offsets for time-series modeling
train_df = df[df['location'] == sample_loc].copy().sort_values('date')
min_date = train_df['date'].min()
train_df['day_offset'] = (train_df['date'] - min_date).dt.days

X = train_df[['day_offset']].values
y = train_df['temperature'].values

# Fit 2nd Degree Polynomial Regressor
degree = 2
model = make_pipeline(PolynomialFeatures(degree=degree), Ridge(alpha=1.0))
model.fit(X, y)
y_pred = model.predict(X)

r2 = r2_score(y, y_pred)
rmse = np.sqrt(mean_squared_error(y, y_pred))

print(f"Model Performance on {sample_loc}:")
print(f"- R² Score: {r2:.4f}")
print(f"- Root Mean Squared Error (RMSE): {rmse:.2f} °C")

# Future Forecast (Next 365 Days / 12 Months)
last_day = train_df['day_offset'].max()
future_days = np.array([last_day + i * 30 for i in range(1, 13)]).reshape(-1, 1)
future_preds = model.predict(future_days)
future_dates = [train_df['date'].max() + pd.Timedelta(days=i*30) for i in range(1, 13)]

# Plot Historical vs Forecast
plt.figure(figsize=(14, 6))
plt.plot(train_df['date'], y, label='Historical Observed Temp', color='#3b82f6', alpha=0.5)
plt.plot(train_df['date'], y_pred, label='Fitted Polynomial Trend', color='#1e3a8a', linewidth=2)
plt.plot(future_dates, future_preds, label='Projected 12-Month Trend (Ridge)', color='#ef4444', linewidth=2.5, linestyle='--')
plt.fill_between(future_dates, future_preds - 1.96 * rmse, future_preds + 1.96 * rmse, color='#ef4444', alpha=0.2, label='95% Confidence Interval')

plt.title(f'12-Month Predictive Climate Temperature Trend for {sample_loc}', fontsize=14, fontweight='bold')
plt.xlabel('Date')
plt.ylabel('Temperature (°C)')
plt.legend()
plt.tight_layout()
plt.show()

### 8. Exporting Tableau-Ready Processed Datasets

In [ ]:
export_dir = os.path.join('..', 'data', 'processed')
os.makedirs(export_dir, exist_ok=True)
export_path = os.path.join(export_dir, 'earthscape_tableau_master.csv')

df.to_csv(export_path, index=False)
print(f"Successfully generated Tableau-ready dataset: {export_path}")